# Module 3: Data: the real job

Run the cells top to bottom. Cells marked **YOUR TURN** are the build: they run as they are, but they won't pass the done-when checks until you fill them in. All data in this kit is synthetic.

In [ ]:
# Runs only on Google Colab: fetch the starter kit and its packages, and copy any keys
# you saved in Colab's Secrets panel into the environment. Locally this does nothing.
import os, sys
if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/Go-Everyway/ai-track-starter /content/ai-track-starter
    %cd /content/ai-track-starter
    %pip install -q -r requirements.txt
    from google.colab import userdata
    for name in ["LLM_API_KEY", "LLM_PROVIDER", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL", "LLM_DAILY_BUDGET_USD"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set in Secrets; lib/llm.py uses its default

In [ ]:
# Find the repo root so paths work from notebooks/ or from Colab.
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "leads_clean.csv").exists())
DATA = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("starter kit at", ROOT)

## 1. Look before you touch anything

This is the same 4,000 leads as `leads_clean.csv`, the way a CRM actually exports them.
Count the problems first. Every fix below is a decision, and `decisions` keeps a record of
each one so someone else can check your work.

In [ ]:
messy = pd.read_csv(DATA / "leads_messy.csv", dtype=str)
decisions = []

print(f"{len(messy):,} rows, {messy.shape[1]} columns")
print("\nblank cells per column:")
print(messy.isna().sum()[messy.isna().sum() > 0].to_string())
print(f"\nindustry has {messy.industry.nunique()} different spellings")
print(f"country has {messy.country.nunique()} different spellings")
print(f"lead_source has {messy.lead_source.nunique()} different spellings")

## 2. Personal data first

The export carries contact names and emails. A model doesn't need either. The email is still
useful for one job, spotting the same person entered twice, so swap it for a keyed hash
(HMAC: a stand-in that's the same every time for the same email, so duplicates still match,
but can't be read back without the secret key), then drop both raw columns. A hashed email is
still personal data under GDPR: hashing cuts the exposure, it doesn't remove the obligation.
Do this before the data goes anywhere else, including any AI tool.

In [ ]:
import hashlib
import hmac

HASH_KEY = b"change-me-and-keep-it-out-of-git"  # a secret: in real use, load it from .env
messy["contact_key"] = messy.contact_email.str.strip().str.lower().map(
    lambda e: hmac.new(HASH_KEY, e.encode(), hashlib.sha256).hexdigest()[:16]
)
messy = messy.drop(columns=["contact_name", "contact_email"])
decisions.append("pii: replaced contact_email with a keyed hash (HMAC, contact_key) for dedupe; dropped contact_name and contact_email")

## 3. Dates in three formats

`2025-03-14`, `03/14/2025` and `14 Mar 2025` all mean the same day. Parse each format
explicitly instead of letting pandas guess, because a guess can swap day and month.

In [ ]:
def parse_date(s):
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d %b %Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT

messy["created_date"] = messy.created_date.map(parse_date)
print("unparsed dates:", messy.created_date.isna().sum())
decisions.append("dates: parsed ISO, US (mm/dd/yyyy) and '14 Mar 2025' formats explicitly; no guessing")

## 4. YOUR TURN: duplicates

The same contact shows up again under a new lead id, with the company typed a little
differently. Match on the contact key, keep the first entry (the lowest lead id), drop the
rest. Do it before you split: a duplicate that lands in both train and test lets the model
see test answers during training.

In [ ]:
from sklearn.model_selection import train_test_split

dupes = messy.contact_key.duplicated(keep=False)
print(f"{messy.contact_key.duplicated().sum()} extra rows belong to contacts already in the file")
print(messy[dupes].sort_values("contact_key")[["lead_id", "company", "lead_source", "converted"]].head(6).to_string())

# What would happen without dedupe: how many duplicate pairs straddle a random split?
tr, te = train_test_split(messy, test_size=0.25, random_state=0, stratify=messy.converted)
straddle = len(set(tr.contact_key) & set(te.contact_key))
print(f"\nwithout dedupe, {straddle} contacts would sit in both train and test")

# Keep one row per contact_key: sort by lead_id so the first-entered row comes first, then
# drop the later copies. One line of pandas.
messy = messy.reset_index(drop=True)
print(f"after dedupe: {len(messy):,} rows")
decisions.append("TODO: how you matched duplicates, which copy you kept, and how many rows you dropped")

## 5. YOUR TURN: one spelling per category

Normalize case and spacing, then map every variant to one name. Write the map out by hand.
It's tedious, and it's the only way to be sure "Consulting" really means professional services.

`INDUSTRY` is done for you as the example. Fill in `COUNTRY` and `SOURCE`: run the cell, read
the list of unmapped variants it prints, and add each one to the right map until the list is empty.
Every key is lower case.

In [ ]:
INDUSTRY = {
    "wholesale distribution": "Wholesale distribution", "wholesale": "Wholesale distribution", "distribution": "Wholesale distribution",
    "manufacturing": "Manufacturing", "mfg": "Manufacturing", "manufacturer": "Manufacturing",
    "food & beverage": "Food & beverage", "food and beverage": "Food & beverage", "f&b": "Food & beverage",
    "logistics": "Logistics", "3pl": "Logistics", "logistics & warehousing": "Logistics",
    "retail": "Retail", "retail - consumer": "Retail", "ecommerce": "Retail",
    "healthcare": "Healthcare", "health care": "Healthcare", "medical": "Healthcare",
    "construction": "Construction", "construction & trades": "Construction",
    "professional services": "Professional services", "prof. services": "Professional services", "consulting": "Professional services",
    "software": "Software", "saas": "Software", "tech - software": "Software",
    "education": "Education", "edtech": "Education", "training": "Education",
}
COUNTRY = {
    "united states": "United States",  # add the rest
}
SOURCE = {
    "inbound_demo": "inbound_demo",  # add the rest
}

def normalize(col, mapping):
    key = messy[col].str.strip().str.lower()
    unmapped = sorted(set(key.dropna()) - set(mapping))
    if unmapped:
        print(f"{col}: still unmapped: {unmapped}")
    return key.map(mapping).fillna(key)  # unmapped variants pass through until you map them

for col, mapping in [("industry", INDUSTRY), ("country", COUNTRY), ("lead_source", SOURCE)]:
    messy[col] = normalize(col, mapping)
    print(f"{col}: {messy[col].nunique()} values")
decisions.append("TODO: how you made each category one spelling")

## 6. YOUR TURN: numbers stored as text, and numbers that can't be true

`employees` arrives as `97`, `1,200` or a range like `51-200`. Strip the commas, take the
middle of a range, and treat 999,999 employees as a typo, not a giant company.
`pages_viewed` has a couple of `-1`s, which can't happen.

In [ ]:
def parse_employees(v):
    # Right now anything that isn't a plain number becomes blank. Handle "1,200" and "51-200".
    return pd.to_numeric(v, errors="coerce")

messy["employees"] = messy.employees.map(parse_employees)
impossible = messy.employees > 100_000
print(f"{impossible.sum()} impossible employee counts set to blank")
messy.loc[impossible, "employees"] = float("nan")
decisions.append("TODO: how you turned employees into a number, and what you did with ranges and impossible values")

for c in ["pages_viewed", "pricing_page_visits", "email_opens", "email_clicks", "webinar_attended",
          "hiring_ops_roles", "recent_funding", "meetings_held", "converted"]:
    messy[c] = pd.to_numeric(messy[c])
bad_pages = messy.pages_viewed < 0
messy.loc[bad_pages, "pages_viewed"] = float("nan")
print(f"{bad_pages.sum()} negative page counts set to blank")
decisions.append("pages_viewed: negative values set to blank as entry errors")

## 7. Blanks: pick a fix and know what it assumes

- **Drop the row:** assumes blanks are random. They rarely are, and here you'd lose 997 of 4,120 rows.
- **Fill numbers with the median:** assumes a blank lead is typical. Add a `_missing` flag so the
  model can still learn if blank-ness itself means something.
- **Fill categories with "unknown":** says "we don't know" out loud instead of guessing.

In [ ]:
print("blanks before:", messy[["industry", "employees", "country", "pages_viewed", "erp_type"]].isna().sum().to_dict())
for c in ["employees", "pages_viewed"]:
    messy[f"{c}_missing"] = messy[c].isna().astype(int)
    messy[c] = messy[c].fillna(messy[c].median())
for c in ["industry", "country", "erp_type"]:
    messy[c] = messy[c].fillna("unknown")
print("blanks after:", int(messy.isna().sum().sum()))
decisions.append("missing numbers: median fill plus a _missing flag column; missing categories: 'unknown' as its own value")

## 8. YOUR TURN: hunt for leakage

Score every numeric column on its own with Module 1's catch rate. A real signal catches
maybe 20% to 30% of conversions by itself. Anything near 100% is too good to be true,
and too good to be true almost always means the column was recorded after the outcome.

In [ ]:
import numpy as np

def catch_rate(scores, converted):
    """Share of all conversions that land in the top-scored 10% of leads."""
    scores, converted = np.asarray(scores, dtype=float), np.asarray(converted)
    k = max(1, len(scores) // 10)
    tiebreak = np.random.default_rng(0).random(len(scores))
    order = np.lexsort((tiebreak, -scores))
    return converted[order[:k]].sum() / converted.sum()

numeric = ["employees", "pages_viewed", "pricing_page_visits", "email_opens", "email_clicks",
           "webinar_attended", "hiring_ops_roles", "recent_funding", "meetings_held"]
alone = pd.Series({c: catch_rate(messy[c], messy.converted) for c in numeric}).sort_values(ascending=False)
print(alone.map("{:.1%}".format).to_string())

In [ ]:
print(pd.crosstab(messy.meetings_held, messy.converted, margins=True))

In [ ]:
# Which column records the outcome instead of predicting it? Look at the table above, then the
# crosstab, and ask when each column gets filled in.
leaking_column = None
why_it_leaks = """
"""

## 9. Retrain, with and without the leak

Same split as Module 1. Logistic regression this time, which needs its numbers on similar
scales, so standardize them. `class_weight="balanced"` tells it to care about the rare
converters as much as the common non-converters.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORIES = ["industry", "country", "lead_source", "seniority", "job_function", "erp_type"]
ALL_NUMBERS = ["employees", "pages_viewed", "pricing_page_visits", "email_opens", "email_clicks",
               "webinar_attended", "hiring_ops_roles", "recent_funding", "meetings_held",
               "employees_missing", "pages_viewed_missing"]

def score(train, test, numbers):
    model = make_pipeline(
        ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
                           ("num", StandardScaler(), numbers)]),
        LogisticRegression(max_iter=2000, class_weight="balanced"),
    ).fit(train, train.converted)
    return catch_rate(model.predict_proba(test)[:, 1], test.converted)

train, test = train_test_split(messy, test_size=0.25, random_state=0, stratify=messy.converted)
with_leak = score(train, test, ALL_NUMBERS)
without_leak = score(train, test, [c for c in ALL_NUMBERS if c != leaking_column])
results = pd.Series({"every column": with_leak, f"without {leaking_column or '(nothing yet)'}": without_leak})
print(results.map("{:.1%}".format).to_string())
decisions.append("TODO: which column you dropped as a leak")

In [ ]:
# YOUR TURN: two sentences. Why is the lower score the one to report?
why_lower_is_true = """
"""

## 10. Your record

In [ ]:
for i, d in enumerate(decisions, 1):
    print(f"{i}. {d}")

## Done-when checks

In [ ]:
from lib.checks import done_when

_ = done_when([
    ("No contact names or emails left in the data", not {"contact_name", "contact_email"} & set(messy.columns)),
    ("Every contact appears once", not messy.contact_key.duplicated().any()),
    ("One spelling per category (10 industries plus 'unknown')", messy.industry.nunique() == 11 and messy.lead_source.nunique() == 7),
    ("No blanks, no impossible values", messy.isna().sum().sum() == 0 and messy.employees.max() <= 100_000 and messy.pages_viewed.min() >= 0),
    ("Every cleaning choice is written down", sum(not d.startswith("TODO") for d in decisions) >= 8),
    ("You found the leaking column", leaking_column == "meetings_held"),
    ("You explained why it leaks", len(why_it_leaks.split()) >= 20),
    ("Before and after scores, and the leak inflated the score", with_leak > without_leak),
    ("You explained why the lower score is the true one", len(why_lower_is_true.split()) >= 20),
])